# Generate Model B training images
Select the adversarial checkpoint with the lowest recorded total generator loss, process only persisted training images, preserve class folders, and save a complete source manifest. The reserved test split is not generated or accessed here.

In [ ]:
from pathlib import Path

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import load_splits
from applied_ai_midterm.generation import generate_training_images
from applied_ai_midterm.models import Generator
from applied_ai_midterm.training import load_best_generator, select_device
from applied_ai_midterm.visualization import plot_srgan_comparisons

In [ ]:
config = load_config(Path("configs/config.yaml"))
device = select_device()
splits = load_splits(Path("data/splits"))
generator = Generator()
checkpoint_path, checkpoint = load_best_generator(
    generator, Path("checkpoints/srgan"), device
)
print(f"Selected epoch {checkpoint['epoch']} from {checkpoint_path}")
print(f"Generator selection loss: {checkpoint.get('generator_selection_loss')}")

In [ ]:
# Only splits.train is passed. Generated files are grouped by detected class_name.
manifest = generate_training_images(
    generator,
    splits.train,
    Path("data/generated"),
    Path("data/generated/manifest.csv"),
    checkpoint_path,
    device,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
display(manifest.head())
display(manifest.groupby(["class_name", "label"]).size().rename("generated_count").to_frame())
assert len(manifest) == len(splits.train)
assert set(manifest.source_filepath).isdisjoint(set(splits.test.filepath))

In [ ]:
comparison_figure = plot_srgan_comparisons(
    generator,
    splits.train,
    device,
    output_path=Path("artifacts/srgan_generation_comparison.png"),
    max_samples=4,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
comparison_figure